<a href="https://colab.research.google.com/github/keilladesantana28/Aula-de-Array/blob/main/C%C3%B3pia_de_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Árvores Binárias de Busca (BST) — Tutorial

**COMP0497 — Algoritmos e Estruturas de Dados 1**  
Aula 08

## Objetivos

Ao final deste tutorial você será capaz de:

- Implementar a propriedade da BST em código e verificá-la;
- Programar travessias in-order, pre-order e post-order;
- Implementar busca, mínimo, máximo, sucessor, inserção e remoção;
- Visualizar BSTs e comparar a altura em diferentes ordens de inserção;
- Comparar a implementação Python com a referência em Java moderno (`../codigo-java/BST.java`).

> O código de produção da aula está em **Java com generics**. Aqui usamos Python apenas para visualização rápida e exercícios.

## 1. Propriedade da BST

Para todo nó `x`: todas as chaves na subárvore esquerda são `<= x.key` e todas na direita são `>= x.key`. Isso vale **recursivamente** em cada nó.

Vamos definir um nó simples e verificar a propriedade.

In [ ]:
# Exemplo: nó de BST e verificação da propriedade
from dataclasses import dataclass, field
from typing import Optional, Any

@dataclass
class Node:
    key: Any
    value: Any = None
    left:  Optional['Node'] = None
    right: Optional['Node'] = None

def is_bst(x: Optional[Node], lo=float('-inf'), hi=float('inf')) -> bool:
    if x is None:
        return True
    if not (lo <= x.key <= hi):
        return False
    return is_bst(x.left, lo, x.key) and is_bst(x.right, x.key, hi)

# Árvore da Fig. 12.1(a) de Cormen: chaves {2,3,5,5,7,8}
root = Node(5,
    left=Node(3, left=Node(2), right=Node(5)),
    right=Node(7, right=Node(8)))
print('is_bst?', is_bst(root))

In [ ]:
import random, math
# Exercício 1: BST inválida
# TODO: monte uma árvore que VIOLA a propriedade da BST e mostre que is_bst retorna False.
#       Dica: insira um nó com chave maior que a raiz na subárvore esquerda.

from dataclasses import dataclass, field
from typing import Optional, Any

@dataclass
class Node:

    key: Any
    value: Any = None
    left:  Optional['Node'] = None
    right: Optional['Node'] = None

def is_bst(x: Optional[Node], lo=float('-inf'), hi=float('inf')) -> bool:
    if x is None:
        return True

    if not (lo <= x.key <= hi):
        return False

    return is_bst(x.left, lo, x.key) and is_bst(x.right, x.key, hi)

# Example of a valid BST (from previous cell, for testing the fixed is_bst)
root = Node(5,
    left=Node(3, left=Node(2), right=Node(5)),
    right=Node(7, right=Node(8)))
print('is_bst? (valid tree)', is_bst(root))

# TODO: construa uma árvore que viola a propriedade

ruim = Node(5,
            left=Node(6, left=Node(2), right=Node(3)), # Viola:  6 é filho de 5 e está na direita
            right=Node(7, right=Node(8)))

print('is_bst? (invalid tree)', is_bst(ruim))
assert is_bst(ruim) is False, 'A árvore deveria violar a propriedade'

is_bst? (valid tree) True
is_bst? (invalid tree) False


## 2. Travessias

Três travessias recursivas naturais. A **in-order** imprime as chaves em ordem crescente.

In [ ]:
# Exemplo: travessias em ordem, pre-ordem e pós-ordem
def inorder(x, out):
    if x is None: return
    inorder(x.left, out)
    out.append(x.key)
    inorder(x.right, out)

def preorder(x, out):
    if x is None: return
    out.append(x.key)
    preorder(x.left, out)
    preorder(x.right, out)

def postorder(x, out):
    if x is None: return
    postorder(x.left, out)
    postorder(x.right, out)
    out.append(x.key)

i, p, q = [], [], []
inorder(root, i); preorder(root, p); postorder(root, q)
print('in-order  :', i)   # ordenado
print('pre-order :', p)
print('post-order:', q)

in-order  : [2, 3, 5, 5, 7, 8]
pre-order : [5, 3, 2, 5, 7, 8]
post-order: [2, 5, 3, 8, 7, 5]


In [ ]:
import random, math
# Exercício 2: in-order iterativo com pilha
# TODO: implemente in-order SEM recursão usando uma pilha.

from dataclasses import dataclass, field
from typing import Optional, Any

@dataclass
class Node:
    key: Any
    value: Any = None
    left:  Optional['Node'] = None
    right: Optional['Node'] = None


root = Node(5,
    left=Node(3, left=Node(2), right=Node(5)),
    right=Node(7, right=Node(8)))

def inorder_iter(raiz):
  out = []
  stack = []
  current = raiz
  while current or stack:
    while current:
      stack.append(current)
      current = current.left
    current = stack.pop()
    out.append(current.key)
    current = current.right
  return out

saida = inorder_iter(root)
print('in-order (iterativa):' , saida )
assert saida == [2, 3, 5, 5, 7, 8], 'Verifique sua implementação'


in-order (iterative): [2, 3, 5, 5, 7, 8]


## 3. BST completa: busca, inserção, mínimo, sucessor

Vamos implementar uma BST genérica em Python, espelhando a `BST.java` da aula.

In [ ]:
# Exemplo: BST completa estilo Sedgewick (inserção recursiva)


def _node_size(node):
    return node.size if node else 0

class BST:
    def __init__(self):
        self.root = None

    def get(self, key):
        x = self.root
        while x is not None:
            if   key < x.key: x = x.left
            elif key > x.key: x = x.right
            else: return x.value
        return None

    def put(self, key, value=None):
        self.root = self._put(self.root, key, value)

    def _put(self, x, key, value):
        if x is None: return Node(key, value)
        if   key < x.key: x.left  = self._put(x.left,  key, value)
        elif key > x.key: x.right = self._put(x.right, key, value)
        else:             x.value = value

        x.size = 1 + _node_size(x.left) + _node_size(x.right)
        return x

    def min(self):
        x = self.root
        while x.left is not None: x = x.left
        return x.key

    def keys_inorder(self):
        out = []; inorder(self.root, out); return out

    def size(self):
        return _node_size(self.root)

    def height(self):
        def h(x): return -1 if x is None else 1 + max(h(x.left), h(x.right))
        return h(self.root)

arv = BST()
for k in [15, 6, 18, 3, 7, 17, 20, 13, 9]:
    arv.put(k, f'v{k}')
print('in-order :', arv.keys_inorder())
print('min      :', arv.min())
print('altura   :', arv.height())
print('get(13)  :', arv.get(13))
print('get(42)  :', arv.get(42))

in-order : [3, 6, 7, 9, 13, 15, 17, 18, 20]
min      : 3
altura   : 4
get(13)  : v13
get(42)  : None


In [ ]:
# Visualização: desenhar a BST com matplotlib
import matplotlib.pyplot as plt

def desenhar(no, x=0.0, y=0.0, dx=2.0, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4)); ax.set_axis_off()
    if no is None: return ax
    ax.add_patch(plt.Circle((x, y), 0.25, fill=True, color='#cfe2ff', ec='#0d6efd'))
    ax.text(x, y, str(no.key), ha='center', va='center', fontsize=10, fontweight='bold')
    if no.left:
        ax.plot([x, x - dx], [y - 0.25, y - 1 + 0.25], color='#0d6efd')
        desenhar(no.left,  x - dx, y - 1, dx / 1.8, ax)
    if no.right:
        ax.plot([x, x + dx], [y - 0.25, y - 1 + 0.25], color='#0d6efd')
        desenhar(no.right, x + dx, y - 1, dx / 1.8, ax)
    return ax

ax = desenhar(arv.root); ax.set_aspect('equal'); plt.tight_layout(); plt.show()

NameError: name 'arv' is not defined

In [ ]:
# Exercício 3: sucessor in-order
# TODO: implemente a função sucessor(arvore, chave) que retorna a próxima
#       chave em ordem crescente, ou None se 'chave' for o máximo.
#       Casos: (a) se o nó tem subárvore direita, sucessor = mínimo dela;
#              (b) caso contrário, sobe pelos pais até encontrar um nó
#                  cujo filho esquerdo também seja ancestral do nó alvo.
# Dica: sem ponteiro para o pai em Python, você pode procurar a partir da raiz
#       guardando o último ancestral onde você foi para a esquerda.
import random, math
from dataclasses import dataclass, field
from typing import Optional, Any


@dataclass
class Node:
    key: Any
    value: Any = None
    left:  Optional['Node'] = None
    right: Optional['Node'] = None
    size: int = 1

    def __str__(self):
        return f'Node({self.key})'


def inorder(x, out):
    if x is None: return
    inorder(x.left, out)
    out.append(x.key)
    inorder(x.right, out)


def _node_size(node):
    return node.size if node else 0


class BST:
    def __init__(self):
        self.root = None

    def get(self, key):
        x = self.root
        while x is not None:
            if   key < x.key: x = x.left
            elif key > x.key: x = x.right
            else: return x.value
        return None

    def put(self, key, value=None):
        self.root = self._put(self.root, key, value)

    def _put(self, x, key, value):
        if x is None: return Node(key, value)
        if   key < x.key: x.left  = self._put(x.left,  key, value)
        elif key > x.key: x.right = self._put(x.right, key, value)
        else: x.value = value

        x.size = 1 + _node_size(x.left) + _node_size(x.right)
        return x

    def min(self):
        x = self.root
        while x.left is not None: x = x.left
        return x.key

    def keys_inorder(self):
        out = []; inorder(self.root, out); return out

    def height(self):
        def h(x): return -1 if x is None else 1 + max(h(x.left), h(x.right))
        return h(self.root)


def inorder_iter(raiz):
  out = []
  stack = []
  current = raiz
  while current or stack:
    while current:
      stack.append(current)
      current = current.left
    current = stack.pop()
    out.append(current.key)
    current = current.right
  return out

def _sucessor(tree: BST, key):
    # TODO: implemente
    current_node = tree.root
    successor_val = None

    while current_node is not None:
        if key < current_node.key:
            successor_val = current_node.key
            current_node = current_node.left
        elif key > current_node.key:
            current_node = current_node.right
        else:
            if current_node.right is not None:

                temp_node = current_node.right
                while temp_node.left is not None:
                    temp_node = temp_node.left
                return temp_node.key
            else:

                return successor_val

    return None


arv = BST()
for k in [15, 6, 18, 3, 7, 17, 20, 13, 9]:
    arv.put(k, f'v{k}')



assert _sucessor(arv, 13) == 15, f"Expected successor of 13 to be 15, but got {_sucessor(arv, 13)}"
assert _sucessor(arv, 15) == 17, f"Expected successor of 15 to be 17, but got {_sucessor(arv, 15)}"
assert _sucessor(arv, 20) is None, f"Expected successor of 20 to be None, but got {_sucessor(arv, 20)}"

print('sucessor(13)  :', _sucessor(arv, 13))
print('sucessor(15)  :', _sucessor(arv, 15))
print('sucessor(20)  :', _sucessor(arv, 20))

sucessor(13)  : 15
sucessor(15)  : 17
sucessor(20)  : None


## 4. Remoção (Hibbard)

Três casos: 0 filhos, 1 filho, 2 filhos (substitui pelo sucessor).

In [ ]:
# Exemplo: remoção de Hibbard
def _min(x): return x if x.left is None else _min(x.left)

def _delete_min(x):
    if x.left is None: return x.right
    x.left = _delete_min(x.left); return x

def delete(x, key):
    if x is None: return None
    if   key < x.key: x.left  = delete(x.left,  key)
    elif key > x.key: x.right = delete(x.right, key)
    else:
        if x.right is None: return x.left
        if x.left  is None: return x.right
        t = x; x = _min(t.right); x.right = _delete_min(t.right); x.left = t.left
    return x

arv.root = delete(arv.root, 6)   # 6 tem 2 filhos: caso 3
print('após delete(6):', arv.keys_inorder())
ax = desenhar(arv.root); ax.set_aspect('equal'); plt.tight_layout(); plt.show()

In [ ]:
# Exercício 4: contar nós em cada subárvore (size)
# TODO: implemente size(no) que retorna o número de nós da subárvore.
#       Em seguida, modifique put para manter o campo size em cada nó
#       (como a BST.java de referência).



from dataclasses import dataclass
from typing import Optional, Any

@dataclass
class Node:
    key: Any
    value: Any = None
    left: Optional['Node'] = None
    right: Optional['Node'] = None
    size: int = 1  # importante

class BST:
    def __init__(self):
        self.root = None

    def size(self, no=None):
        # Se no for None: tamanho da árvore toda.
           # Se no for um nó: tamanho da subárvore desse nó.
        if no is None:
            return 0
        # Se 'no' é um Node, seu atributo 'size' é mantido atualizado pelo _put.
        return no.size

    def put(self, key, value=None):
        self.root = self._put(self.root, key, value)

    def _put(self, x, key, value):
        if x is None:
            return Node(key, value)

        if key < x.key:
            x.left = self._put(x.left, key, value)
        elif key > x.key:
            x.right = self._put(x.right, key, value)
        else:
            x.value = value  # chave já existe, só atualiza valor

        # mantém size correto
        # Aqui, self.size(x.left) e self.size(x.right) são chamadas para obter o size atualizado das subárvores
        x.size = 1 + self.size(x.left) + self.size(x.right)
        return x

    def height(self):
        def h(x):
            return -1 if x is None else 1 + max(h(x.left), h(x.right))
        return h(self.root)



arv = BST()
for k in [15, 6, 18, 3, 7, 17, 20, 13]:
    arv.put(k)

assert arv.size(None) == 0
assert arv.size(arv.root) == 8   # A árvore é populada com 8 elementos
print("OK")

OK


## 5. Altura: ordem de inserção importa

Vamos comparar a altura de uma BST construída com chaves **aleatórias** vs **ordenadas**.

In [ ]:
# Visualização: altura cresce com n, em log para aleatório, linear para ordenado
import random, math

def altura_para(n, ordem):
    arv = BST()
    for k in ordem: arv.put(k)
    return arv.height()

ns = [10, 50, 100, 200, 500, 1000, 2000]
alts_rand, alts_ord = [], []
for n in ns:
    chaves = list(range(n))
    alts_ord.append(altura_para(n, chaves))
    random.seed(42); random.shuffle(chaves)
    alts_rand.append(altura_para(n, chaves))

plt.figure(figsize=(7, 4))
plt.plot(ns, alts_rand, 'o-', label='ordem aleatória')
plt.plot(ns, alts_ord,  's-', label='ordem crescente (pior caso)')
plt.plot(ns, [2*math.log2(n) for n in ns], 'k--', alpha=0.5, label='2·log₂(n)')
plt.xlabel('n (número de chaves)'); plt.ylabel('altura h')
plt.legend(); plt.title('Altura da BST vs ordem de inserção'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Desafio Final

Implemente uma **agenda telefônica** usando a BST genérica `BST.java` (em `../codigo-java/`). A agenda deve:

1. Inserir contatos (`nome -> telefone`), mantendo unicidade do nome;
2. Buscar telefone por nome;
3. Listar contatos em ordem alfabética;
4. Listar contatos cujo nome começa com uma letra dada (use sucessor in-order);
5. Remover um contato.

Adicionalmente, **meça a altura** da BST resultante após inserir 1000 nomes aleatórios e compare com $\lceil 2 \log_2 1000 \rceil \approx 20$. Discuta:

- O que aconteceria se você inserisse os nomes em ordem alfabética?
- Como uma árvore balanceada (próxima aula) resolveria isso?

In [ ]:
# Desafio: Agenda telefônica com BST
# TODO: implemente a classe Agenda usando uma BST como armazenamento interno.
#       Você pode reusar a classe BST definida acima.

class Agenda:
    def __init__(self):
        self._arv = BST() # A instância da BST é armazenada em _arv

    def inserir(self, nome: str, telefone: str):
        # Delega a inserção para a BST interna
        self._arv.put(nome, telefone)




    def buscar(self, nome: str):

        return self._arv.get(nome)

    def listar_em_ordem(self):
        # listagem em ordem para a BST interna
        return self._arv.keys_inorder()

    # As funções _min e _delete_min são auxiliares para a remoção da BST e estão definidas globalmente.
    # A classe Agenda deve chamar a função de remoção que opera na raiz da BST.

    def remover(self, nome: str):
        # A função 'delete' (definida globalmente) modifica a raiz da BST.
        # Precisamos atualizar a raiz da BST interna da Agenda.
        global delete # Garante que a função global delete seja referenciada
        self._arv.root = delete(self._arv.root, nome)

# Teste mínimo
a = Agenda()
for n, t in [('Carla', '99'), ('Ana', '11'), ('Bruno', '55'), ('Diego', '77')]:
    a.inserir(n, t)
assert a.listar_em_ordem() == ['Ana', 'Bruno', 'Carla', 'Diego']
assert a.buscar('Bruno') == '55'

print(a.listar_em_ordem())

['Ana', 'Bruno', 'Carla', 'Diego']


## Referências

Veja `../referencias.bib`. Em particular:

- **Cormen et al. (2009)**, *Introduction to Algorithms*, cap. 12 — pseudocódigo e provas de complexidade.
- **Sedgewick (2003)**, *Algorithms in Java, Parts 1–4*, cap. 12 — estilo da implementação recursiva em Java usado em `../codigo-java/BST.java`.